# Errors and recovery demo

Every cell below fails on purpose, in a different way. Outputs are checked in, so
the tracebacks render without a kernel.

This is the fixture for the error-handling behavior that still needs work
(roadmap KR-01 and KR-03): how failures render, whether partial output survives,
and what the agent sees when it reads a failed cell.

## stdout and stderr in one cell

Two separate `stream` outputs, in order. They should be visually distinguishable —
stderr is conventionally tinted.

In [1]:
import sys

print("stdout: starting reconciliation")
sys.stderr.write("stderr: 3 records could not be matched\n")
print("stdout: finished with warnings")

stdout: starting reconciliation
stdout: finished with warnings


stderr: 3 records could not be matched


## A warning

`warnings.warn` routes through stderr and carries the source location.

In [2]:
import warnings


def parse_legacy(payload):
    warnings.warn("parse_legacy() is deprecated; use parse_v2()", DeprecationWarning, stacklevel=2)
    return {"value": payload}


parse_legacy("42")

/var/folders/qz/kjb0dbb56jlgbcy9mnch0p4h0000gp/T/ipykernel_64501/228130222.py:9: DeprecationWarning: parse_legacy() is deprecated; use parse_v2()
  parse_legacy("42")


{'value': '42'}

## Partial output, then a failure

The important case: the cell printed real work before it died. That output must
not be discarded — both in the rendered notebook and in
`jupyter.get_cell_output`.

In [3]:
totals = []
for index, divisor in enumerate([5, 4, 3, 0, 2]):
    print(f"step {index}: dividing 100 by {divisor}")
    totals.append(100 / divisor)

print("never reached")

step 0: dividing 100 by 5
step 1: dividing 100 by 4
step 2: dividing 100 by 3
step 3: dividing 100 by 0


ZeroDivisionError: division by zero

## A deep traceback

Several frames, so the traceback is long enough to test scrolling and collapse.

In [4]:
def load_config(path):
    return read_section(path, "runtime")


def read_section(path, section):
    return coerce_port(path, section)


def coerce_port(path, section):
    raw = {"runtime": {"port": "not-a-number"}}[section]
    return int(raw["port"])


load_config("/etc/nimbalyst/config.toml")

ValueError: invalid literal for int() with base 10: 'not-a-number'

## Missing key

A `KeyError` from pandas, which renders differently from a plain dict lookup.

In [5]:
import pandas as pd

frame = pd.DataFrame({"service": ["api-gateway"], "deploys": [12]})
frame["latency_ms"]

KeyError: 'latency_ms'

## Failed assertion

Custom message, no stack beyond the call site.

In [6]:
def check_invariant(deploys, rollbacks):
    assert rollbacks <= deploys, f"rollbacks ({rollbacks}) cannot exceed deploys ({deploys})"
    return deploys - rollbacks


check_invariant(3, 7)

AssertionError: rollbacks (7) cannot exceed deploys (3)

## The cell after the failures

This one succeeds. Run-all should reach it or stop before it, and whichever the
extension does needs to be deliberate — an agent reading this notebook has to be
able to tell "not run yet" apart from "ran and failed".

In [7]:
print("recovered: this cell is independent of everything above")

recovered: this cell is independent of everything above


## For the agent

- `jupyter.get_cell_output` on the partial-output cell should return the three
  printed lines *and* the error.
- `jupyter.list_cells` reports `stale` / `executedBeforeRestart` flags, which are
  null here because these outputs were produced in an earlier session.
- `jupyter.run_all` on this notebook is the honest test of how failures are
  reported mid-run.